# 09 — Spatial Validation

## Objective

This notebook evaluates the spatial characteristics of the 30 m LST downscaled products using additional spatial validation measures.

The analysis is designed to determine whether Random Forest and XGBoost preserve meaningful spatial structure in the downscaled LST products rather than simply producing smoother temperature distributions.

### Analyses

1. Moran's I — spatial autocorrelation
2. Variogram analysis — spatial dependence across distance
3. Hotspot preservation — spatial agreement of thermal extremes
4. Thermal variability and smoothing assessment

## 1. Moran's I — Spatial Autocorrelation

In [30]:
!pip install libpysal esda


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import numpy as np
from pathlib import Path
import rasterio
from libpysal.weights import lat2W
from esda.moran import Moran

In [36]:
PROJECT_DIR = Path(r"D:\Projects\GeoAI-LST-Downscaling")
COARSE_DIR = PROJECT_DIR / "data" / "processed" / "lucknow" / "coarse"

reference_path = PROJECT_DIR / "data" / "processed" / "lucknow" / "fine" / "lst_clean_lucknow.tif"
tsharp_path = COARSE_DIR / "tsharp_lst_90m_lucknow.tif"
rf_path = COARSE_DIR / "rf_lst_90m_lucknow.tif"
xgb_path = COARSE_DIR / "xgb_lst_90m_lucknow.tif"

In [37]:




def read_raster(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Raster not found: {path}")

    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float64)
        profile = src.profile.copy()
        nodata = src.nodata

    # Convert NoData values to NaN
    if nodata is not None and not np.isnan(nodata):
        data[data == nodata] = np.nan

    return data, profile, nodata

reference, ref_profile, ref_nodata = read_raster(reference_path)
tsharp, tsharp_profile, tsharp_nodata = read_raster(tsharp_path)
rf, rf_profile, rf_nodata = read_raster(rf_path)
xgb, xgb_profile, xgb_nodata = read_raster(xgb_path)

print("Reference:", reference.shape)
print("TsHARP:", tsharp.shape)
print("RF:", rf.shape)
print("XGBoost:", xgb.shape)

FileNotFoundError: Raster not found: D:\Projects\GeoAI-LST-Downscaling\data\processed\lucknow\fine\lst_clean_lucknow.tif

In [38]:
print("Reference shape:", reference.shape)
print("TsHARP shape:", tsharp.shape)
print("RF shape:", rf.shape)
print("XGBoost shape:", xgb.shape)

print("Reference resolution:", ref_profile["transform"].a, abs(ref_profile["transform"].e))
print("TsHARP resolution:", tsharp_profile["transform"].a, abs(tsharp_profile["transform"].e))
print("RF resolution:", rf_profile["transform"].a, abs(rf_profile["transform"].e))
print("XGBoost resolution:", xgb_profile["transform"].a, abs(xgb_profile["transform"].e))

Reference shape: (703, 762)
TsHARP shape: (234, 254)
RF shape: (234, 254)
XGBoost shape: (234, 254)
Reference resolution: 30.0 30.0
TsHARP resolution: 90.0 90.0
RF resolution: 90.0 90.0
XGBoost resolution: 90.0 90.0


In [35]:
if not (
    reference.shape == tsharp.shape == rf.shape == xgb.shape
):
    raise ValueError("All rasters must have the same shape and grid.")

valid_mask = (
    np.isfinite(reference)
    & np.isfinite(tsharp)
    & np.isfinite(rf)
    & np.isfinite(xgb)
)

print("Valid pixels:", valid_mask.sum())

ValueError: All rasters must have the same shape and grid.

In [ ]:
valid_mask &= reference != -9999
valid_mask &= tsharp != -9999
valid_mask &= rf != -9999
valid_mask &= xgb != -9999

In [ ]:
reference_values = reference[valid_mask]
tsharp_values = tsharp[valid_mask]
rf_values = rf[valid_mask]
xgb_values = xgb[valid_mask]

print("Number of valid pixels:", len(reference_values))

Number of valid pixels: 35383


In [ ]:
from libpysal.weights import lat2W

rows, cols = reference.shape

# Rook contiguity: edge-sharing neighbours
w = lat2W(rows, cols, rook=True)
# Create rook-contiguity weights for the complete raster grid
w_full = lat2W(rows, cols, rook=True)
print("Number of spatial units:", w.n)
print("Number of neighbours for first pixel:", len(w.neighbors[0]))

Number of spatial units: 59436
Number of neighbours for first pixel: 2


In [ ]:
print("Total pixels:", reference.size)
print("Valid pixels:", valid_mask.sum())
print("Invalid pixels:", (~valid_mask).sum())

Total pixels: 59436
Valid pixels: 35383
Invalid pixels: 24053


In [ ]:
valid_flat = valid_mask.flatten()

valid_indices = np.where(valid_flat)[0]

print("Valid spatial units:", len(valid_indices))

Valid spatial units: 35383


In [ ]:
from libpysal.weights import W

valid_indices = np.where(valid_flat)[0]

# Map original raster index -> new index
index_map = {
    old_idx: new_idx
    for new_idx, old_idx in enumerate(valid_indices)
}

neighbors = {}
weights = {}

for old_idx in valid_indices:
    new_idx = index_map[old_idx]

    # Original neighbours from the full raster
    old_neighbors = w_full.neighbors[old_idx]

    # Keep only neighbours that are also valid
    valid_neighbors = [
        n for n in old_neighbors
        if n in index_map
    ]

    neighbors[new_idx] = [
        index_map[n] for n in valid_neighbors
    ]

    weights[new_idx] = [1.0] * len(valid_neighbors)

# Create the reduced spatial weights object
w = W(neighbors, weights)

print("Original spatial units:", w_full.n)
print("Valid spatial units:", len(valid_indices))
print("Spatial units in new weights:", w.n)

Original spatial units: 59436
Valid spatial units: 35383
Spatial units in new weights: 35383


In [ ]:
print("Number of islands:", w.n_components)
print("Average neighbours:", np.mean(list(map(len, w.neighbors.values()))))

Number of islands: 1
Average neighbours: 3.9651244948138937


In [ ]:
reference_values = reference[valid_mask]
tsharp_values = tsharp[valid_mask]
rf_values = rf[valid_mask]
xgb_values = xgb[valid_mask]

In [ ]:
from esda.moran import Moran

# Calculate Moran's I
moran_reference = Moran(
    reference_values,
    w,
    permutations=999
)

moran_tsharp = Moran(
    tsharp_values,
    w,
    permutations=999
)

moran_rf = Moran(
    rf_values,
    w,
    permutations=999
)

moran_xgb = Moran(
    xgb_values,
    w,
    permutations=999
)

In [ ]:
print("Reference 90 m")
print("Moran's I:", moran_reference.I)
print("Expected I:", moran_reference.EI)
print("Z-score:", moran_reference.z_sim)
print("Permutation p-value:", moran_reference.p_sim)

print("\nTsHARP 90 m")
print("Moran's I:", moran_tsharp.I)
print("Expected I:", moran_tsharp.EI)
print("Z-score:", moran_tsharp.z_sim)
print("Permutation p-value:", moran_tsharp.p_sim)

print("\nRandom Forest 90 m")
print("Moran's I:", moran_rf.I)
print("Expected I:", moran_rf.EI)
print("Z-score:", moran_rf.z_sim)
print("Permutation p-value:", moran_rf.p_sim)

print("\nXGBoost 90 m")
print("Moran's I:", moran_xgb.I)
print("Expected I:", moran_xgb.EI)
print("Z-score:", moran_xgb.z_sim)
print("Permutation p-value:", moran_xgb.p_sim)

Reference 90 m
Moran's I: 0.9313515287299169
Expected I: -2.8262958566502742e-05
Z-score: 249.52133145508995
Permutation p-value: 0.001

TsHARP 90 m
Moran's I: 0.9313515287299169
Expected I: -2.8262958566502742e-05
Z-score: 237.82791952171806
Permutation p-value: 0.001

Random Forest 90 m
Moran's I: 0.8597083541798675
Expected I: -2.8262958566502742e-05
Z-score: 224.40071639956162
Permutation p-value: 0.001

XGBoost 90 m
Moran's I: 0.7855861370814959
Expected I: -2.8262958566502742e-05
Z-score: 203.88750278075145
Permutation p-value: 0.001


In [ ]:
print("Reference vs TsHARP identical:",
      np.array_equal(reference_values, tsharp_values))

print("Maximum absolute difference:",
      np.max(np.abs(reference_values - tsharp_values)))

print("Mean absolute difference:",
      np.mean(np.abs(reference_values - tsharp_values)))

Reference vs TsHARP identical: True
Maximum absolute difference: 0.0
Mean absolute difference: 0.0
